# Storage Mapper — Missing Tank Pilot (Colab GPU · DOTA Aerial OBB)

Uses an open aerial/satellite model (**YOLO26-OBB / YOLO11-OBB**, pretrained on DOTA where `storage tank` is a native top-down class). Everything runs remotely in Colab — nothing is downloaded locally.

1. Select **Runtime -> Change runtime type -> T4 GPU** (free tier).
2. Click **Runtime -> Run all**.

- **Green circles:** Already mapped on OpenStreetMap
- **Red dashed circles:** Unmapped tank candidates detected by the aerial model
- **Fast re-tuning:** Tiles stay cached in Colab RAM, so after the first run you can adjust `CONF_THRESH` in Cell 1 and re-run only Cells 1, 4, and 5.

In [ ]:
#@title 1. Pilot Settings
MODEL_NAME = "yolo26x-obb.pt" #@param ["yolo26x-obb.pt", "yolo11x-obb.pt", "yolo26l-obb.pt", "yolo11l-obb.pt"]
COUNTRY_ISO2 = "IQ" #@param ["IQ", "SY", "IR", "TR", "KW", "SA", "JO", "AE"]
PILOT_LIMIT = 5 #@param {type:"integer"}
MIN_FARM_TANKS = 4 #@param {type:"integer"}
MAX_FARM_TANKS = 25 #@param {type:"integer"}
CONF_THRESH = 0.35 #@param {type:"slider", min:0.15, max:0.85, step:0.05}
ZOOM = 18 #@param [17, 18] {type:"raw"}
BUFFER_M = 200 #@param {type:"integer"}
MAX_TILES_PER_FARM = 100 #@param {type:"integer"}
DOWNLOAD_GEOJSON = False #@param {type:"boolean"}

print(f"Config ready -> model={MODEL_NAME} | country={COUNTRY_ISO2} | pilot_farms={PILOT_LIMIT} | conf={CONF_THRESH} | zoom={ZOOM}")


In [ ]:
#@title 2. Install Ultralytics & Load Pretrained DOTA Aerial Model
%pip install -q ultralytics

import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print(f"GPU active: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU (works fine for 5 pilot farms; switch Runtime -> T4 GPU for full-country speed).")

model = YOLO(MODEL_NAME)
tank_ids = [k for k, v in model.names.items() if "storage" in v.lower() and "tank" in v.lower()]
if not tank_ids:
    raise RuntimeError(f"Could not locate 'storage tank' class in model names: {model.names}")
TANK_CLS_ID = tank_ids[0]
print(f"Loaded {MODEL_NAME} | target class {TANK_CLS_ID}: '{model.names[TANK_CLS_ID]}'")


In [ ]:
#@title 3. Fetch OSM Storage Tanks & Cluster Into Farms
import math, json, time, requests
import numpy as np
from scipy.spatial import cKDTree

OVERPASS_ENDPOINTS = [
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
]

HEADERS = {
    "User-Agent": "StorageMapper-Pilot/2.0 (+https://github.com/rawazrauf/storage_mapper)",
    "Accept": "application/json",
}

QUERY = f"""[out:json][timeout:120];
area["ISO3166-1"="{COUNTRY_ISO2}"][admin_level=2]->.searchArea;
way["man_made"="storage_tank"](area.searchArea);
out geom tags;"""

def fetch_tanks():
    for url in OVERPASS_ENDPOINTS:
        host = url.split("/")[2]
        for attempt in range(2):
            try:
                t0 = time.time()
                r = requests.post(url, data={"data": QUERY}, headers=HEADERS, timeout=120)
                if r.status_code == 429 and attempt == 0:
                    wait = min(int(r.headers.get("Retry-After", 8)), 15)
                    print(f"Rate-limited on {host} (429), retrying in {wait}s...")
                    time.sleep(wait)
                    continue
                r.raise_for_status()
                data = r.json()
                if "elements" in data:
                    print(f"Fetched {len(data['elements'])} ways from {host} in {time.time()-t0:.1f}s")
                    return data["elements"]
            except Exception as e:
                print(f"Endpoint failed ({host}): {e}")
                break
    raise RuntimeError("All Overpass endpoints failed. Re-run this cell in a moment.")

def tank_metrics(geom):
    n = len(geom)
    lats = [p["lat"] for p in geom]
    lons = [p["lon"] for p in geom]
    lat_avg = sum(lats) / n
    lon_avg = sum(lons) / n
    cos_lat = math.cos(math.radians(lat_avg))
    area, perim = 0.0, 0.0
    for i in range(n - 1):
        x0, y0 = geom[i]["lon"], geom[i]["lat"]
        x1, y1 = geom[i + 1]["lon"], geom[i + 1]["lat"]
        area += x0 * y1 - x1 * y0
        dx = (x1 - x0) * cos_lat * 111320.0
        dy = (y1 - y0) * 111320.0
        perim += math.hypot(dx, dy)
    area = abs(area) / 2.0 * (111320.0 ** 2) * cos_lat
    circ = (4 * math.pi * area) / (perim ** 2) if perim > 0 else 0.0
    diam = 2.0 * math.sqrt(area / math.pi) if area > 0 else 0.0
    return lat_avg, lon_avg, circ, diam

raw = fetch_tanks()
all_osm_tanks = []  # used to mask ALL already-mapped tanks (including <6m)
tanks = []          # circular >=6m tanks used for farm clustering

for el in raw:
    geom = el.get("geometry", [])
    if len(geom) < 3:
        continue
    lat, lon, circ, diam = tank_metrics(geom)
    rec = {"id": el["id"], "lat": lat, "lon": lon, "diam_m": max(diam, 2.0), "geom": geom}
    all_osm_tanks.append(rec)
    if circ >= 0.75 and diam >= 6.0:
        tanks.append(rec)

mid_lat = float(np.mean([t["lat"] for t in tanks]))
cos_mid = math.cos(math.radians(mid_lat))
pts_m = np.array([[t["lat"] * 111320.0, t["lon"] * 111320.0 * cos_mid] for t in tanks])
tree = cKDTree(pts_m)
parent = list(range(len(tanks)))

def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

for i, j in tree.query_pairs(r=500.0):
    ri, rj = find(i), find(j)
    if ri != rj:
        parent[ri] = rj

groups = {}
for idx, t in enumerate(tanks):
    groups.setdefault(find(idx), []).append(t)

clusters = sorted(groups.values(), key=len, reverse=True)
eligible = [c for c in clusters if MIN_FARM_TANKS <= len(c) <= MAX_FARM_TANKS]
print(f"All OSM tanks (masking): {len(all_osm_tanks)} | Circular >=6m: {len(tanks)} across {len(clusters)} clusters")
print(f"Eligible farms ({MIN_FARM_TANKS}-{MAX_FARM_TANKS} tanks): {len(eligible)} | Pilot will scan: {min(PILOT_LIMIT, len(eligible)) if PILOT_LIMIT > 0 else len(eligible)}")


In [ ]:
#@title 4. Run Sliding-Window YOLO-OBB Detection, Audit Recall & Export GeoJSON
import io
from PIL import Image
import matplotlib.pyplot as plt

TILE_URL = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
WIN_PX = 1024
STRIDE_PX = 768
MIN_ASPECT = 0.65   # reject elongated non-cylindrical boxes
MIN_DIAM_M = 5.0

session = requests.Session()
session.headers.update({"User-Agent": "StorageMapper-Pilot/2.0"})

if "tile_cache" not in globals():
    tile_cache = {}

def latlon_to_world_px(lat, lon, z):
    n = 2 ** z
    x = (lon + 180.0) / 360.0 * n * 256.0
    lat_r = math.radians(lat)
    y = (1.0 - math.asinh(math.tan(lat_r)) / math.pi) / 2.0 * n * 256.0
    return x, y

def world_px_to_latlon(px, py, z):
    n = 2 ** z
    lon = (px / 256.0) / n * 360.0 - 180.0
    lat = math.degrees(math.atan(math.sinh(math.pi * (1.0 - 2.0 * (py / 256.0) / n))))
    return lat, lon

def fetch_tile(z, tx, ty):
    key = (z, tx, ty)
    if key in tile_cache:
        return tile_cache[key]
    for _ in range(2):
        try:
            r = session.get(TILE_URL.format(z=z, x=tx, y=ty), timeout=20)
            r.raise_for_status()
            arr = np.array(Image.open(io.BytesIO(r.content)).convert("RGB"))
            tile_cache[key] = arr
            time.sleep(0.03)
            return arr
        except Exception:
            time.sleep(0.3)
    return np.zeros((256, 256, 3), dtype=np.uint8)

def fetch_mosaic(cluster):
    lats = [t["lat"] for t in cluster]
    lons = [t["lon"] for t in cluster]
    dlat = BUFFER_M / 111320.0
    dlon = BUFFER_M / (111320.0 * math.cos(math.radians(float(np.mean(lats)))))
    x0, y1 = latlon_to_world_px(min(lats) - dlat, min(lons) - dlon, ZOOM)
    x1, y0 = latlon_to_world_px(max(lats) + dlat, max(lons) + dlon, ZOOM)

    tx0, tx1 = int(x0 // 256), int(x1 // 256)
    ty0, ty1 = int(y0 // 256), int(y1 // 256)
    nx, ny = tx1 - tx0 + 1, ty1 - ty0 + 1
    if nx * ny > MAX_TILES_PER_FARM:
        print(f"Skipping oversized cluster ({nx}x{ny}={nx*ny} tiles > {MAX_TILES_PER_FARM})")
        return None, 0, 0

    mosaic = np.zeros((ny * 256, nx * 256, 3), dtype=np.uint8)
    for ty in range(ty0, ty1 + 1):
        for tx in range(tx0, tx1 + 1):
            py, px = (ty - ty0) * 256, (tx - tx0) * 256
            mosaic[py:py+256, px:px+256] = fetch_tile(ZOOM, tx, ty)
    return mosaic, tx0 * 256, ty0 * 256

def window_starts(total, win, stride):
    if total <= win:
        return [0]
    starts = list(range(0, total - win + 1, stride))
    if starts[-1] != total - win:
        starts.append(total - win)
    return starts

def scan_cluster_obb(cluster, cid):
    mosaic, ox, oy = fetch_mosaic(cluster)
    if mosaic is None:
        return None, [], [], 0
    H, W = mosaic.shape[:2]
    lat_mid = float(np.mean([t["lat"] for t in cluster]))
    m_per_px = 156543.03392 * math.cos(math.radians(lat_mid)) / (2 ** ZOOM)

    # All mapped OSM tanks inside this mosaic
    known_all = []
    for t in all_osm_tanks:
        wx, wy = latlon_to_world_px(t["lat"], t["lon"], ZOOM)
        lx, ly = wx - ox, wy - oy
        if -40 <= lx <= W + 40 and -40 <= ly <= H + 40:
            known_all.append((lx, ly, (t["diam_m"] / 2.0) / m_per_px))

    # Sliding-window inference at native DOTA 1024px scale
    raw_dets = []
    for y0 in window_starts(H, WIN_PX, STRIDE_PX):
        for x0 in window_starts(W, WIN_PX, STRIDE_PX):
            crop = mosaic[y0:min(y0 + WIN_PX, H), x0:min(x0 + WIN_PX, W)]
            preds = model.predict(
                Image.fromarray(crop),
                imgsz=WIN_PX,
                conf=CONF_THRESH,
                classes=[TANK_CLS_ID],
                device=DEVICE,
                verbose=False,
            )
            obb = preds[0].obb
            if obb is None or len(obb) == 0:
                continue
            xywhr = obb.xywhr.cpu().numpy()
            confs = obb.conf.cpu().numpy()
            for (cx, cy, w, h, _), conf in zip(xywhr, confs):
                mx = max(w, h)
                if mx <= 0 or (min(w, h) / mx) < MIN_ASPECT:
                    continue
                r_px = 0.25 * (w + h)
                if (r_px * 2.0 * m_per_px) < MIN_DIAM_M:
                    continue
                raw_dets.append((float(conf), float(x0 + cx), float(y0 + cy), float(r_px)))

    # Cross-window NMS
    raw_dets.sort(reverse=True, key=lambda d: d[0])
    dedup = []
    for conf, cx, cy, rpx in raw_dets:
        if any(math.hypot(cx - hx, cy - hy) < max(rpx, hr) * 0.85 for _, hx, hy, hr in dedup):
            continue
        dedup.append((conf, cx, cy, rpx))

    # Separate into known-OSM recall matches vs. genuinely unmapped candidates
    matched_known_ids = set()
    candidates = []
    for conf, cx, cy, rpx in dedup:
        overlapped = False
        for kidx, (ex, ey, er) in enumerate(known_all):
            if math.hypot(cx - ex, cy - ey) < max(rpx + er * 0.6, er * 1.05, rpx * 1.05):
                matched_known_ids.add(kidx)
                overlapped = True
        if overlapped:
            continue
        lat, lon = world_px_to_latlon(ox + cx, oy + cy, ZOOM)
        candidates.append({
            "cluster_id": cid,
            "score": round(conf, 3),
            "lat": lat,
            "lon": lon,
            "diam_m": round(rpx * 2.0 * m_per_px, 1),
            "cx": cx,
            "cy": cy,
            "r_px": rpx
        })

    return mosaic, known_all, candidates, len(matched_known_ids)

def circle_poly(lat, lon, diam_m, n=20):
    r_m = diam_m / 2.0
    dlat = r_m / 111320.0
    dlon = r_m / (111320.0 * math.cos(math.radians(lat)))
    ring = [[round(lon + dlon * math.cos(2 * math.pi * i / n), 7),
             round(lat + dlat * math.sin(2 * math.pi * i / n), 7)] for i in range(n)]
    ring.append(ring[0])
    return ring

target_clusters = [c for c in clusters if MIN_FARM_TANKS <= len(c) <= MAX_FARM_TANKS]
if PILOT_LIMIT and PILOT_LIMIT > 0:
    target_clusters = target_clusters[:PILOT_LIMIT]

all_candidates = []
scanned_clusters = []
show_plots = (PILOT_LIMIT is not None and 0 < PILOT_LIMIT <= 15)

for cid, cl in enumerate(target_clusters):
    t0 = time.time()
    mosaic, known_all, cands, matched_cnt = scan_cluster_obb(cl, cid)
    if mosaic is None:
        continue
    scanned_clusters.append((cid, cl))
    all_candidates.extend(cands)
    c_lat = float(np.mean([t["lat"] for t in cl]))
    c_lon = float(np.mean([t["lon"] for t in cl]))
    recall_pct = (100.0 * matched_cnt / len(known_all)) if known_all else 0.0
    print(f"Farm #{cid} ({c_lat:.4f}, {c_lon:.4f}) [{time.time()-t0:.1f}s] | Known OSM recall: {matched_cnt}/{len(known_all)} ({recall_pct:.0f}%) | NEW candidates: {len(cands)}")

    if show_plots:
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(mosaic)
        for ex, ey, er in known_all:
            ax.add_patch(plt.Circle((ex, ey), er, fill=False, edgecolor="#4ade80", lw=1.4))
        for c in cands:
            ax.add_patch(plt.Circle((c["cx"], c["cy"]), c["r_px"], fill=False, edgecolor="#ef4444", lw=2.2, ls="--"))
            ax.text(c["cx"], c["cy"] - c["r_px"] - 4, f"{c['score']:.2f}", color="yellow",
                    fontsize=8, ha="center", weight="bold", bbox=dict(fc="black", alpha=0.6, pad=1, lw=0))
        ax.set_title(f"Farm #{cid} ({c_lat:.4f}, {c_lon:.4f}) | OSM recall {matched_cnt}/{len(known_all)} | Red=New ({len(cands)})")
        ax.axis("off")
        plt.show()

all_candidates.sort(key=lambda c: c["score"], reverse=True)
geojson = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "geometry": {"type": "Polygon", "coordinates": [circle_poly(c["lat"], c["lon"], c["diam_m"])]},
        "properties": {
            "score": c["score"],
            "est_diam_m": c["diam_m"],
            "cluster_id": c["cluster_id"],
            "coords": f"{c['lat']:.5f}, {c['lon']:.5f}",
            "osm_url": f"https://www.openstreetmap.org/#map=18/{c['lat']:.5f}/{c['lon']:.5f}",
            "josm_url": f"http://127.0.0.1:8111/load_and_zoom?left={c['lon']-0.002:.5f}&right={c['lon']+0.002:.5f}&top={c['lat']+0.002:.5f}&bottom={c['lat']-0.002:.5f}"
        }
    } for c in all_candidates]
}

out_path = "missing_tanks_pilot.geojson"
with open(out_path, "w") as f:
    json.dump(geojson, f)
print()
print(f"Saved {len(all_candidates)} ranked candidates to {out_path} (z{ZOOM} tiles cached in RAM: {len(tile_cache)})")

if DOWNLOAD_GEOJSON and len(all_candidates) > 0:
    from google.colab import files
    files.download(out_path)


In [ ]:
#@title 5. Interactive Map (Esri Satellite + 1-Click JOSM Links)
import folium
from IPython.display import display, HTML

if not all_candidates:
    print("No unmapped candidates found above CONF_THRESH. Either these 5 farms are already well-mapped, or try lowering CONF_THRESH to 0.25 in Cell 1.")
else:
    print("Top unmapped candidates (sorted by confidence):")
    for i, c in enumerate(all_candidates[:25], 1):
        print(f"{i:2d}. Farm #{c['cluster_id']:2d} | conf={c['score']:.3f} | diam~{c['diam_m']:4.1f}m | {c['lat']:.5f}, {c['lon']:.5f}")

    top = all_candidates[0]
    m = folium.Map(location=[top["lat"], top["lon"]], zoom_start=17, tiles=None, max_zoom=19)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Imagery",
        name="Esri Satellite",
        max_native_zoom=18,
        max_zoom=19
    ).add_to(m)

    for cid, cl in scanned_clusters:
        for t in cl:
            pts = [[p["lat"], p["lon"]] for p in t["geom"]]
            folium.Polygon(
                locations=pts, color="#4ade80", weight=1.5, fill=True, fill_opacity=0.2,
                tooltip=f"OSM Tank #{t['id']} (~{t['diam_m']:.0f}m)"
            ).add_to(m)

    for c in all_candidates:
        ring = circle_poly(c["lat"], c["lon"], c["diam_m"])
        pts = [[lat, lon] for lon, lat in ring]
        josm = f"http://127.0.0.1:8111/load_and_zoom?left={c['lon']-0.002:.5f}&right={c['lon']+0.002:.5f}&top={c['lat']+0.002:.5f}&bottom={c['lat']-0.002:.5f}"
        osm = f"https://www.openstreetmap.org/#map=18/{c['lat']:.5f}/{c['lon']:.5f}"
        html = (
            f"<b>Unmapped Candidate (Farm #{c['cluster_id']})</b><br>"
            f"Confidence: <b>{c['score']:.3f}</b><br>"
            f"Est. diameter: ~{c['diam_m']:.1f} m<br>"
            f"Coords: <code>{c['lat']:.5f}, {c['lon']:.5f}</code><br><br>"
            f"<a href='{josm}' target='_blank'>Open in JOSM (Remote Control)</a><br>"
            f"<a href='{osm}' target='_blank'>View on OpenStreetMap</a>"
        )
        folium.Polygon(
            locations=pts, color="#ef4444", weight=2.5, dash_array="5,5",
            fill=True, fill_opacity=0.15,
            tooltip=f"Conf {c['score']:.2f} (~{c['diam_m']:.0f}m)",
            popup=folium.Popup(html, max_width=260)
        ).add_to(m)

    display(HTML(m._repr_html_()))
